# MM-Net — external validation on Sleep-EDF

Every result so far is internal to iSLEEPS: ten-fold cross-validation is patient-
independent, but it is still one corpus, one centre and one acquisition protocol. This
notebook asks the question that cross-validation cannot answer — does the model transfer
to a cohort it has never seen, recorded elsewhere, on different hardware, from different
people?

**Protocol.** Train once on all 96 iSLEEPS patients, then run inference on Sleep-EDF
Expanded with **no dataset-specific tuning**: no fine-tuning, no threshold search, no
re-fitted normalisation, and the HMM transition matrix estimated from iSLEEPS only.

**What this can and cannot test.** Sleep-EDF carries EEG, EOG and EMG, so the staging
head is fully exercised. It carries no SpO2, ECG, effort or pulse channels and no
respiratory-event annotations, so the respiratory head cannot be evaluated at all — the
cardiorespiratory features are set to zero. That is not a confound for staging: our
ablation showed removing the entire cardiorespiratory stream leaves staging unchanged
(Wilcoxon p = 0.91).

**The confound that does apply** is montage. Sleep-EDF is frontal/occipital (Fpz-Cz,
Pz-Oz); iSLEEPS is central/occipital (C4:M1, C3:M2, O2:M1, O1:M2). Any drop mixes three
causes — different patients, different montage, different equipment — and we cannot
separate them with one external corpus. We report the drop and name the confound rather
than attributing it.

In [1]:
import glob
import json
import os
import sys
import time

import numpy as np
from sklearn.metrics import (accuracy_score, cohen_kappa_score, confusion_matrix,
                             f1_score)

REPO = os.path.abspath(os.path.join(os.getcwd(), "..", "..", "..", ".."))
sys.path.insert(0, os.path.join(REPO, "MMNet_research", "model"))
import mmnet_core as C  # noqa: E402

EXT = os.path.join(REPO, "data", "sleep_edf_mm")
OUT = os.path.join(REPO, "MMNet_research", "results", "revision", "runs")
os.makedirs(OUT, exist_ok=True)
print("device:", C.DEV, "| iSLEEPS patients:", len(C.SUBS))
print("external recordings:", len(glob.glob(os.path.join(EXT, "*.npz"))))

cwd: D:\sleep-staging-psg\MMNet_research\MMNet_Submission\all_codes\notebooks | device: cuda | NVIDIA GeForce RTX 2060


subjects: 96 (SN28 dropped) | epochs: 89,532
stage %: {'W': np.float64(26.6), 'N1': np.float64(10.2), 'N2': np.float64(42.3), 'N3': np.float64(8.9), 'R': np.float64(12.1)}
respiratory-event prevalence: 16.0%
EEG-feature counts -> EEG: 112 EOG: 50 EMG: 26
parameters (concat): 773,254
training utilities defined.
device: cuda | iSLEEPS patients: 96
external recordings: 11


## 1. Load the external cohort

Normalisation is per-recording z-scoring, identical to the iSLEEPS loader. This is the
one operation applied to the external data, and it uses only that recording's own
statistics — no iSLEEPS constants and no test-set information leak between recordings.

In [2]:
EXTDATA = {}
for f in sorted(glob.glob(os.path.join(EXT, "*.npz"))):
    rec = os.path.basename(f)[:-4]
    d = np.load(f)
    Fe = np.nan_to_num(d["Feeg"]).astype(np.float32)
    Fc = np.nan_to_num(d["Fcard"]).astype(np.float32)
    Fe = (Fe - Fe.mean(0)) / (Fe.std(0) + 1e-6)      # same as the iSLEEPS loader
    EXTDATA[rec] = (Fe, Fc, d["y"].astype(np.int64))

n_ep = sum(len(v[2]) for v in EXTDATA.values())
yall = np.concatenate([v[2] for v in EXTDATA.values()])
sc = np.bincount(yall, minlength=5)
print("external: %d recordings, %d epochs" % (len(EXTDATA), n_ep))
print("stage %%:", {c: round(100 * sc[i] / n_ep, 1) for i, c in enumerate(C.CLS)})

isc = np.bincount(np.concatenate([C.DATA[s][2] for s in C.SUBS]), minlength=5)
itot = isc.sum()
print("iSLEEPS  %%:", {c: round(100 * isc[i] / itot, 1) for i, c in enumerate(C.CLS)})

external: 11 recordings, 11272 epochs
stage %%: {'W': np.float64(16.3), 'N1': np.float64(8.8), 'N2': np.float64(44.6), 'N3': np.float64(13.0), 'R': np.float64(17.3)}
iSLEEPS  %%: {'W': np.float64(26.6), 'N1': np.float64(10.2), 'N2': np.float64(42.3), 'N3': np.float64(8.9), 'R': np.float64(12.1)}


## 2. Train once on the full iSLEEPS cohort

No held-out iSLEEPS patients are needed, because the test set is an entirely separate
corpus. A small validation split is retained purely for early stopping.

In [3]:
t0 = time.time()
rng = np.random.RandomState(42)
subs = list(C.SUBS); rng.shuffle(subs)
nv = max(10, len(subs) // 9)
va, tr = subs[:nv], subs[nv:]
print("train %d patients | early-stopping split %d" % (len(tr), len(va)))

model = C.train_fold(tr, va, "concat", [], [], seed=42)

# HMM transition matrix from iSLEEPS training patients only
Am = np.ones((C.NC, C.NC)); pi = np.ones(C.NC)
for s in tr:
    y = C.DATA[s][2]; pi[y[0]] += 1
    for a, b in zip(y[:-1], y[1:]): Am[a, b] += 1
A_log = np.log(Am / Am.sum(1, keepdims=True)); pi_log = np.log(pi / pi.sum())
print("trained in %.1f min" % ((time.time() - t0) / 60))

train 86 patients | early-stopping split 10


trained in 0.5 min


## 3. Zero-shot inference on Sleep-EDF

In [4]:
yt_all, yp_all, per_rec = [], [], {}
for rec, (Fe, Fc, y) in EXTDATA.items():
    sp, _ = C.infer_arrays(model, Fe, Fc, len(y))
    pred = C.hmm(A_log, pi_log, np.log(sp + C.EPS))
    per_rec[rec] = dict(n=int(len(y)), acc=float(accuracy_score(y, pred)),
                        kappa=float(cohen_kappa_score(y, pred)))
    yt_all.append(y); yp_all.append(pred)

YT, YP = np.concatenate(yt_all), np.concatenate(yp_all)
ext = dict(acc=float(accuracy_score(YT, YP)),
           mf1=float(f1_score(YT, YP, average="macro", zero_division=0)),
           kappa=float(cohen_kappa_score(YT, YP)))

print("%-26s %10s %10s %10s" % ("", "accuracy", "macro-F1", "kappa"))
print("-" * 58)
print("%-26s %10.4f %10.4f %10.4f" % ("iSLEEPS (10-fold, internal)", 0.7227, 0.6510, 0.6106))
print("%-26s %10.4f %10.4f %10.4f" % ("Sleep-EDF (zero-shot)", ext["acc"], ext["mf1"], ext["kappa"]))
print("%-26s %10.4f %10.4f %10.4f" % ("difference", ext["acc"] - 0.7227,
                                       ext["mf1"] - 0.6510, ext["kappa"] - 0.6106))

print("\nper-recording accuracy:")
for rec in sorted(per_rec):
    r = per_rec[rec]
    print("  %-8s n=%5d  acc %.3f  kappa %.3f" % (rec, r["n"], r["acc"], r["kappa"]))
a = [per_rec[r]["acc"] for r in per_rec]
print("  mean %.4f +- %.4f across recordings" % (np.mean(a), np.std(a)))

                             accuracy   macro-F1      kappa
----------------------------------------------------------
iSLEEPS (10-fold, internal)     0.7227     0.6510     0.6106
Sleep-EDF (zero-shot)          0.8002     0.7086     0.7121
difference                     0.0775     0.0576     0.1015

per-recording accuracy:
  SC4001   n=  841  acc 0.725  kappa 0.637
  SC4002   n= 1127  acc 0.779  kappa 0.703
  SC4011   n= 1103  acc 0.820  kappa 0.731
  SC4012   n= 1186  acc 0.837  kappa 0.739
  SC4021   n= 1025  acc 0.807  kappa 0.693
  SC4022   n= 1009  acc 0.720  kappa 0.609
  SC4031   n=  952  acc 0.884  kappa 0.824
  SC4032   n=  911  acc 0.821  kappa 0.741
  SC4042   n= 1200  acc 0.730  kappa 0.615
  SC4051   n=  672  acc 0.763  kappa 0.670
  SC4052   n= 1246  acc 0.885  kappa 0.832
  mean 0.7974 +- 0.0567 across recordings


## 4. Where it transfers and where it breaks

A single accuracy number hides the interesting part. Per-stage recall says which sleep
stages survive the domain shift, and the confusion matrix says what they turn into.

In [5]:
cm = confusion_matrix(YT, YP, labels=range(5))
rec_ = cm.diagonal() / np.maximum(cm.sum(1), 1)
f1s = f1_score(YT, YP, average=None, labels=range(5), zero_division=0)

print("%-5s %8s %10s %10s" % ("stage", "n", "recall", "F1"))
print("-" * 36)
for i, c in enumerate(C.CLS):
    print("%-5s %8d %10.3f %10.3f" % (c, cm.sum(1)[i], rec_[i], f1s[i]))

print("\nrow-normalised confusion (true -> predicted):")
print("      " + "".join("%8s" % c for c in C.CLS))
for i, c in enumerate(C.CLS):
    row = cm[i] / max(cm.sum(1)[i], 1)
    print("%-6s" % c + "".join("%8.3f" % v for v in row))

json.dump({"external": ext, "per_recording": per_rec,
           "per_stage_recall": rec_.tolist(), "per_stage_f1": f1s.tolist(),
           "confusion": cm.tolist()},
          open(os.path.join(OUT, "external_validation_sleepedf.json"), "w"), indent=1)
print("\nwrote external_validation_sleepedf.json")

stage        n     recall         F1
------------------------------------
W         1834      0.925      0.852
N1         997      0.247      0.317
N2        5024      0.943      0.869
N3        1463      0.554      0.691
R         1954      0.781      0.814

row-normalised confusion (true -> predicted):
             W      N1      N2      N3       R
W        0.925   0.023   0.031   0.000   0.021
N1       0.320   0.247   0.291   0.002   0.140
N2       0.006   0.018   0.943   0.015   0.018
N3       0.008   0.000   0.438   0.554   0.000
R        0.047   0.090   0.083   0.000   0.781

wrote external_validation_sleepedf.json


## Reading the result

Whatever the number, it is a genuine out-of-corpus measurement rather than a
cross-validation fold, and it is the first such measurement for this model. Three causes
are entangled in any drop — different patients, a frontal rather than central montage,
and different recording equipment — and one external corpus cannot separate them. The
per-stage breakdown is the more informative half: a stage that survives transfer is
carried by features robust to all three, while a stage that collapses tells us which part
of the representation was corpus-specific.